# NB05 · Semana 10 — DAG de Airflow (punto de despliegue)

**Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07** · Base de la **Entrega 3 (E3)**

> **Todo lo que corre en producción sale de este notebook:** el DAG (`dags/`), el proyecto dbt (`dbt/`) y los docs (`docs/`). Para cambiar algo se cambia la celda correspondiente, se corre el notebook y se commitean los archivos generados.

## Objetivo

Llevar el pipeline (extracción → bronze → dbt → gold) a un **DAG de Airflow**: definir tareas, dependencias, *scheduling* y reintentos. Este notebook no ejecuta Airflow (eso ocurre en el contenedor `airflow-scheduler`): sirve para **construir y validar** el archivo del DAG antes de desplegarlo.

> Base de la **Entrega 3 (E3)**.

## 1. Anatomía de un DAG

Un DAG (grafo acíclico dirigido) declara tareas y el orden entre ellas. Escribimos el DAG a la carpeta `dags/`, que el scheduler de Airflow lee automáticamente.

## 2. Construir el DAG del proyecto

El DAG encadena `preparar_entorno → [validar_postgres, extraer_bronze] → dbt_build → publicar_gold`. Cada tarea hace una sola cosa, así se puede ver dónde falló y reintentar solo esa parte. La celda guarda el código en memoria; la sección 4 lo escribe en `dags/`.

In [1]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import BASE, DAGS_DIR, info
info()
print("DAGs en:", DAGS_DIR)

Entorno detectado: servidor de la cátedra  (grupo: grupo07)
  Lake / bucket   : grupo07   →   bronze en s3://grupo07/bronze/series
  PostgreSQL      : base 'warehouse', usuario 'grupo07', host 'postgres'
  Material en     : /home/jovyan/work
DAGs en: /home/jovyan/work/dags


In [2]:
# El DAG completo. Esta celda solo lo guarda en memoria; la sección 4 lo escribe a dags/.
dag_code = r'''"""
DAG pipeline_grupo07 — URLhaus -> bronze (Delta en MinIO) -> dbt (silver/gold en DuckDB) -> PostgreSQL.

Gold publicada: fct_familias_mensual (atómica: por mes, tipo de amenaza, familia de malware
y tipo de archivo, sacados de los tags de URLhaus), fct_amenazas_mensual (su resumen por tipo)
y fct_features_familias (features por mes y familia, sin fuga temporal).

Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07

Cómo corre en el servidor de la cátedra (Airflow 2.x):
  * El worker de Airflow NO trae deltalake, duckdb ni dbt. La tarea `preparar_entorno`
    crea UNA vez un virtualenv en /tmp/venv_grupo07 con todo lo necesario, y las demás
    tareas corren dentro de ese venv con ExternalPythonOperator (no se reinstala nada
    en cada corrida).
  * Airflow solo ve esta carpeta (dags/ del JupyterHub = /opt/airflow/dags/grupo07/).
    Por eso las credenciales se leen de `dags/.env` y dbt se toma de `dags/dbt/`.
  * Las credenciales se leen DENTRO de cada tarea: no se pasan por op_kwargs ni env=,
    así no quedan visibles en la pestaña "Rendered Template" de la UI de Airflow.

Backfill: cada corrida procesa SOLO su intervalo [data_interval_start, data_interval_end).
  extraer_bronze baja el volcado CSV de URLhaus (últimos 30 días o base completa), filtra
  las URLs registradas en ese intervalo y hace upsert por id en bronze (no borra). Correr dos
  veces el mismo intervalo da el mismo resultado. dbt y publicar_gold recalculan todo
  desde bronze, así que también son idempotentes.

Tareas:
  preparar_entorno -> [validar_postgres, extraer_bronze] -> dbt_build -> publicar_gold
"""
from __future__ import annotations

import os

import pendulum
from airflow.models.dag import DAG
from airflow.models.param import Param
from airflow.operators.bash import BashOperator
from airflow.operators.python import ExternalPythonOperator

AQUI = os.path.dirname(os.path.realpath(__file__))
ENV_FILE = os.path.join(AQUI, ".env")
DBT_SRC = os.path.join(AQUI, "dbt")
DBT_RUN = "/tmp/dbt_run_grupo07"          # copia de trabajo de dbt (dags/ puede ser solo lectura)
VENV = "/tmp/venv_grupo07"
VENV_PY = f"{VENV}/bin/python"
# Si cambiás esta lista, subí VENV_VERSION para que el venv se reconstruya.
VENV_REQS = "dbt-duckdb deltalake pandas pyarrow requests psycopg2-binary"
VENV_VERSION = "v1"


# ----------------------------------------------------------------------------------
# Las funciones de abajo corren dentro del venv (ExternalPythonOperator): Airflow
# copia SOLO el código de la función, así que cada una importa y define lo que usa.
# ----------------------------------------------------------------------------------
def _validar_postgres(env_file):
    """Valida credenciales de PostgreSQL y que haya un esquema donde podamos crear tablas."""
    import os
    import psycopg2

    for linea in open(env_file, encoding="utf-8"):
        linea = linea.strip()
        if linea and not linea.startswith("#") and "=" in linea:
            k, v = linea.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

    con = psycopg2.connect(host=os.environ.get("PGHOST", "postgres"),
                           port=os.environ.get("PGPORT", "5432"),
                           dbname=os.environ.get("PGDATABASE", "warehouse"),
                           user=os.environ["PGUSER"], password=os.environ["PGPASSWORD"],
                           connect_timeout=10)
    with con, con.cursor() as cur:
        cur.execute("select current_user, current_database(), version()")
        usuario, base, version = cur.fetchone()
        print(f"Conectado a '{base}' como '{usuario}' ({version.split(',')[0]})")
        cur.execute("""
            select nspname from pg_namespace
            where has_schema_privilege(nspname, 'CREATE')
              and nspname not like 'pg\\_%' and nspname <> 'information_schema'
            order by (nspname = current_user) desc, (nspname = 'public') desc, nspname
        """)
        esquemas = [r[0] for r in cur.fetchall()]
    con.close()
    pedido = os.environ.get("PG_SCHEMA")
    if pedido and pedido not in esquemas:
        raise RuntimeError(f"PG_SCHEMA={pedido} pero el usuario no puede crear tablas ahí. "
                           f"Esquemas con permiso CREATE: {esquemas}")
    if not esquemas:
        raise RuntimeError("El usuario no tiene permiso CREATE en ningún esquema: pedile a la "
                           "cátedra un esquema propio (p. ej. 'grupo07').")
    print(f"Esquemas donde se pueden crear tablas: {esquemas}. Se usará: {pedido or esquemas[0]}")


def _extraer_bronze(env_file, inicio, fin):
    """Ingesta URLhaus -> bronze (Delta) del intervalo [inicio, fin) que le pasa Airflow.

    Idempotente y apta para backfill: el resultado depende solo del intervalo, no de la
    fecha en que corre. Hace upsert por id: agrega las URLs nuevas, actualiza las que ya
    estaban y no borra ninguna.
    """
    import csv
    import io
    import os
    import time
    import zipfile
    import pandas as pd
    import pyarrow as pa
    import requests
    from deltalake import DeltaTable, write_deltalake
    from deltalake.exceptions import TableNotFoundError

    for linea in open(env_file, encoding="utf-8"):
        linea = linea.strip()
        if linea and not linea.startswith("#") and "=" in linea:
            k, v = linea.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

    bronze = f"s3://{os.environ.get('MINIO_BUCKET', 'grupo07')}/bronze/series"
    opciones = {
        "AWS_ACCESS_KEY_ID": os.environ["AWS_ACCESS_KEY_ID"],
        "AWS_SECRET_ACCESS_KEY": os.environ["AWS_SECRET_ACCESS_KEY"],
        "AWS_ENDPOINT_URL": os.environ.get("AWS_ENDPOINT_URL", "http://minio:9000"),
        "AWS_REGION": os.environ.get("AWS_REGION", "us-east-1"),
        "AWS_ALLOW_HTTP": "true",
        "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    }
    clave = os.environ.get("URLHAUS_AUTH_KEY")
    if not clave:
        raise RuntimeError("Falta URLHAUS_AUTH_KEY en dags/.env")
    cabecera = {"Auth-Key": clave}

    # 1) El intervalo viene del contexto de la corrida, nunca del reloj: el data_interval que
    #    asigna Airflow o, en un reproceso, los parámetros desde/hasta del Trigger.
    def utc(texto):
        t = pd.Timestamp(texto)
        return t.tz_localize("UTC") if t.tzinfo is None else t.tz_convert("UTC")
    desde, hasta = utc(inicio), utc(fin)
    if desde >= hasta:
        raise ValueError(f"Intervalo vacío o al revés: [{desde}, {hasta})")
    print(f"Intervalo a procesar: [{desde}, {hasta})")

    # 2) Fuente. La API /urls/recent/ solo devuelve los últimos 3 días (máx. 1000 URLs), así que
    #    no sirve para un backfill. URLhaus publica dos volcados CSV con todas las columnas:
    #    los últimos 30 días y la base completa. Se usa el más chico que cubre el intervalo.
    #    El reloj solo elige la fuente; el filtro de filas es siempre por el intervalo.
    #    La Auth-Key va dentro de la URL (así lo exige abuse.ch): nunca se imprime.
    ahora = pd.Timestamp.now(tz="UTC")
    base_exports = "https://urlhaus-api.abuse.ch/v2/files/exports/{clave}"
    url_reciente = os.environ.get("URLHAUS_CSV_RECIENTE", base_exports + "/recent.csv").format(clave=clave)
    url_completo = os.environ.get("URLHAUS_CSV_COMPLETO", base_exports + "/full.csv.zip").format(clave=clave)
    if desde >= ahora - pd.Timedelta(days=29):
        url_csv, archivo = url_reciente, "/tmp/urlhaus_grupo07/csv_recent"
    else:
        url_csv, archivo = url_completo, "/tmp/urlhaus_grupo07/csv_completo"

    # El volcado se guarda en /tmp y se reutiliza durante 6 horas: un backfill de muchos días
    # lo descarga una sola vez.
    os.makedirs(os.path.dirname(archivo), exist_ok=True)
    if not os.path.exists(archivo) or time.time() - os.path.getmtime(archivo) > 6 * 3600:
        visible = url_csv.replace(clave, "<AUTH-KEY>")
        # Fallas transitorias (timeout, corte de red, HTTP 429 o 5xx): se reintenta acá adentro,
        # 3 intentos esperando 1 y 2 minutos. Fallas permanentes (401/403 clave inválida, 404/410
        # endpoint retirado, otro 4xx): se corta en el acto; reintentar no lo arregla. Airflow
        # no distingue una de otra, por eso la decisión se toma en la tarea.
        # Los errores de requests incluyen la URL (con la clave): nunca se muestran.
        intentos = 3
        for intento in range(1, intentos + 1):
            print(f"Descargando {visible} (intento {intento}/{intentos})")
            try:
                with requests.get(url_csv, headers=cabecera, timeout=600, stream=True) as res:
                    if res.status_code == 200:
                        with open(archivo + ".parcial", "wb") as f:
                            for bloque in res.iter_content(1 << 20):
                                f.write(bloque)
                        break
                    if res.status_code != 429 and res.status_code < 500:
                        raise RuntimeError(f"URLhaus respondió {res.status_code} al bajar {visible}: "
                                           "falla permanente (clave, endpoint o pedido), no se reintenta")
                    motivo = f"HTTP {res.status_code}"
            except requests.RequestException as e:
                motivo = type(e).__name__
            if intento == intentos:
                raise RuntimeError(f"URLhaus no respondió bien en {intentos} intentos ({motivo}) al bajar {visible}")
            print(f"  falló ({motivo}); reintento en {intento} min")
            time.sleep(60 * intento)
        os.replace(archivo + ".parcial", archivo)

    # 3) Leer el CSV en streaming (la base completa es grande) y escribir en bronze por tandas
    #    mensuales. Formato de URLhaus: líneas de comentario con '#', fechas
    #    'YYYY-MM-DD HH:MM:SS' en UTC. Nunca se tienen en memoria más de 300.000 filas.
    columnas = ['id', 'date_added', 'url', 'url_status', 'last_online',
                'threat', 'tags', 'urlhaus_link', 'reporter']
    esquema = pa.schema([
        ("id", pa.int64()), ("date_added", pa.timestamp("us", tz="UTC")),
        ("url", pa.string()), ("url_status", pa.string()), ("last_online", pa.string()),
        ("threat", pa.string()), ("tags", pa.string()),
        ("urlhaus_link", pa.string()), ("reporter", pa.string()),
    ])
    texto_desde = desde.strftime("%Y-%m-%d %H:%M:%S")
    texto_hasta = hasta.strftime("%Y-%m-%d %H:%M:%S")

    def a_tabla(filas):
        df = pd.DataFrame(filas, columns=columnas)
        df["id"] = df["id"].astype("int64")
        df["date_added"] = pd.to_datetime(df["date_added"], utc=True)
        # tags como los publica URLhaus ("Mozi,elf"), sin espacios de más; silver los separa
        df["tags"] = df["tags"].map(lambda t: ",".join(x.strip() for x in t.split(",") if x.strip()) or None)
        for col in ["url", "url_status", "last_online", "threat", "urlhaus_link", "reporter"]:
            df[col] = df[col].where(df[col].str.strip().ne("") & df[col].ne("None"), None)
        df = df.drop_duplicates(subset=["id"])
        return pa.Table.from_pandas(df, schema=esquema, preserve_index=False)

    try:
        DeltaTable(bronze, storage_options=opciones)
        existe = [True]
    except TableNotFoundError:
        existe = [False]
    escritos = {}           # mes -> URLs escritas en esta corrida

    def volcar(clave_mes, filas):
        # 4) Upsert por id: la URL que ya está en bronze se actualiza (por ejemplo, su estado
        #    online/offline), la nueva se agrega y NINGUNA se borra. El export de URLhaus no es un
        #    histórico completo (de los meses viejos solo trae parte de las URLs), así que
        #    reemplazar el intervalo borraría datos que ya teníamos: pasó el 18/11/2026, ver
        #    docs/INCIDENTE_2026-11-18_backfill.md. Correr dos veces lo mismo deja lo mismo.
        tabla = a_tabla(filas)
        if existe[0]:
            # Se adapta al esquema real de bronze (columna por columna) antes del merge, que
            # exige tipos iguales en origen y destino.
            destino = DeltaTable(bronze, storage_options=opciones).to_pyarrow_dataset().schema
            tabla = tabla.select(destino.names).cast(destino)
        if not existe[0]:
            write_deltalake(bronze, tabla, mode="append", storage_options=opciones)
            existe[0] = True
        else:
            (DeltaTable(bronze, storage_options=opciones)
             .merge(source=tabla, predicate="t.id = s.id", source_alias="s", target_alias="t",
                    merge_schema=True)
             .when_matched_update_all()
             .when_not_matched_insert_all()
             .execute())
        escritos[clave_mes] = escritos.get(clave_mes, 0) + tabla.num_rows

    if zipfile.is_zipfile(archivo):
        zf = zipfile.ZipFile(archivo)
        crudo = zf.open(zf.namelist()[0])
    else:
        crudo = open(archivo, "rb")
    MAX_EN_MEMORIA = 300_000
    pendientes, en_memoria = {}, 0
    with io.TextIOWrapper(crudo, encoding="utf-8", errors="replace", newline="") as lineas:
        sin_comentarios = (l for l in lineas if not l.startswith("#"))
        for r in csv.reader(sin_comentarios):
            if len(r) < len(columnas) or not r[0].strip().isdigit():
                continue
            if texto_desde <= r[1] < texto_hasta:
                pendientes.setdefault(r[1][:7], []).append(r[:len(columnas)])
                en_memoria += 1
                if en_memoria >= MAX_EN_MEMORIA:
                    for clave_mes in sorted(pendientes):
                        volcar(clave_mes, pendientes[clave_mes])
                    pendientes, en_memoria = {}, 0
    for clave_mes in sorted(pendientes):
        volcar(clave_mes, pendientes[clave_mes])

    total = sum(escritos.values())
    if not total:
        print("Nada para escribir en este intervalo (bronze queda como estaba).")
        return 0
    for clave_mes in sorted(escritos):
        print(f"  {clave_mes}: {escritos[clave_mes]} URLs")
    # stg_urlhaus lee los .parquet directamente: se borran los archivos que el merge dejó
    # fuera de la tabla, para que dbt no lea versiones viejas.
    DeltaTable(bronze, storage_options=opciones).vacuum(
        retention_hours=0, enforce_retention_duration=False, dry_run=False)
    print(f"Bronze: {total} URLs del intervalo (upsert por id) en {len(escritos)} mes(es).")
    return total


def _dbt_build(env_file, dbt_src, dbt_run, venv):
    """Copia el proyecto dbt a /tmp y corre `dbt build` (modelos + tests)."""
    import os
    import shutil
    import subprocess

    for linea in open(env_file, encoding="utf-8"):
        linea = linea.strip()
        if linea and not linea.startswith("#") and "=" in linea:
            k, v = linea.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

    shutil.rmtree(dbt_run, ignore_errors=True)
    shutil.copytree(dbt_src, dbt_run, ignore=shutil.ignore_patterns(
        "target", "logs", ".ipynb_checkpoints", "*.duckdb", "*.duckdb.wal"))

    env = dict(os.environ)
    env["HOME"] = "/tmp/home_grupo07"      # DuckDB guarda sus extensiones en $HOME
    os.makedirs(env["HOME"], exist_ok=True)
    env["PATH"] = f"{venv}/bin:" + env.get("PATH", "")
    r = subprocess.run([f"{venv}/bin/dbt", "build", "--profiles-dir", "."],
                       cwd=dbt_run, env=env, capture_output=True, text=True)
    print(r.stdout)
    print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"dbt build terminó con código {r.returncode}")


def _publicar_gold(env_file, dbt_run):
    """Publica las tablas gold de DuckDB en PostgreSQL (lo que lee Metabase)."""
    import os
    import duckdb
    import psycopg2
    from psycopg2.extras import execute_values

    for linea in open(env_file, encoding="utf-8"):
        linea = linea.strip()
        if linea and not linea.startswith("#") and "=" in linea:
            k, v = linea.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

    # Cada tabla gold: columnas a leer de DuckDB (con su tipo) y la clave primaria.
    # El orden de las columnas es el mismo en el select, en el create y en el insert.
    TABLAS = {
        "fct_amenazas_mensual": {
            "columnas": [("anio", "integer"), ("mes", "integer"), ("tipo_amenaza", "text"),
                         ("cantidad_amenazas", "bigint")],
            "clave": ["anio", "mes", "tipo_amenaza"],
        },
        "fct_familias_mensual": {
            "columnas": [("fecha", "date"), ("anio", "integer"), ("mes", "integer"),
                         ("tipo_amenaza", "text"), ("familia_malware", "text"),
                         ("tipo_archivo", "text"), ("cantidad_amenazas", "bigint")],
            "clave": ["anio", "mes", "tipo_amenaza", "familia_malware", "tipo_archivo"],
        },
        # Features (E4): definidos en dbt (fct_features_familias); acá solo se copian
        "fct_features_familias": {
            "columnas": [("fecha", "date"), ("anio", "integer"), ("mes", "integer"),
                         ("familia_malware", "text"), ("cantidad_amenazas", "bigint"),
                         ("cantidad_mes_anterior", "bigint"), ("variacion_mensual_pct", "numeric(14,2)"),
                         ("promedio_movil_3m", "numeric(14,2)"), ("acumulado", "bigint"),
                         ("pct_del_mes", "numeric(5,2)"), ("mes_completo", "boolean")],
            "clave": ["fecha", "familia_malware"],
            "nulos": {"variacion_mensual_pct"},     # vacía si el mes anterior tuvo 0 URLs
        },
    }

    # 1) Leer gold de DuckDB (lo dejó dbt_build)
    ruta = os.path.join(dbt_run, "warehouse.duckdb")
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No existe {ruta}: ¿corrió dbt_build en este mismo worker?")
    datos = {}
    con = duckdb.connect(ruta, read_only=True)
    try:
        for nombre, t in TABLAS.items():
            select = ", ".join(f"cast({c} as {'varchar' if tipo == 'text' else tipo})"
                               for c, tipo in t["columnas"])
            datos[nombre] = con.sql(f"select {select} from {nombre}").fetchall()
            if not datos[nombre]:
                raise RuntimeError(f"{nombre} está vacía en DuckDB: no publico una tabla vacía.")
    finally:
        con.close()

    # Descripciones de schema.yml (dbt las deja en target/manifest.json): se copian como COMMENT
    # en PostgreSQL, así la documentación se escribe una sola vez y Metabase la muestra.
    import json
    docs = {}
    ruta_manifest = os.path.join(dbt_run, "target", "manifest.json")
    if os.path.exists(ruta_manifest):
        for nodo in json.load(open(ruta_manifest, encoding="utf-8"))["nodes"].values():
            if nodo.get("resource_type") == "model" and nodo["name"] in TABLAS:
                docs[nodo["name"]] = nodo

    # 2) Escribir en PostgreSQL en UNA transacción: si algo falla, quedan las versiones anteriores
    pg = psycopg2.connect(host=os.environ.get("PGHOST", "postgres"),
                          port=os.environ.get("PGPORT", "5432"),
                          dbname=os.environ.get("PGDATABASE", "warehouse"),
                          user=os.environ["PGUSER"], password=os.environ["PGPASSWORD"],
                          connect_timeout=10)
    resultado = {}
    try:
        with pg, pg.cursor() as cur:
            esquema = os.environ.get("PG_SCHEMA")
            if not esquema:
                cur.execute("""
                    select nspname from pg_namespace
                    where has_schema_privilege(nspname, 'CREATE')
                      and nspname not like 'pg\\_%' and nspname <> 'information_schema'
                    order by (nspname = current_user) desc, (nspname = 'public') desc, nspname
                    limit 1
                """)
                fila = cur.fetchone()
                if not fila:
                    raise RuntimeError("Sin permiso CREATE en ningún esquema de PostgreSQL.")
                esquema = fila[0]
            for nombre, t in TABLAS.items():
                tabla = f'"{esquema}".{nombre}'
                columnas = ",\n".join(f"{c} {tipo}{'' if c in t.get('nulos', ()) else ' not null'}"
                                       for c, tipo in t["columnas"])
                chequeo = ", check (mes between 1 and 12)" if any(c == "mes" for c, _ in t["columnas"]) else ""
                # Si la tabla ya existe con otras columnas (cambió el modelo), se recrea.
                # Sin CASCADE: si una vista depende de ella, falla y avisa en vez de borrarla.
                cur.execute("""
                    select column_name from information_schema.columns
                    where table_schema = %s and table_name = %s
                """, (esquema, nombre))
                actuales = {r[0] for r in cur.fetchall()}
                esperadas = {c for c, _ in t["columnas"]} | {"actualizado_en"}
                if actuales and actuales != esperadas:
                    print(f"{tabla}: cambiaron las columnas ({sorted(actuales ^ esperadas)}), la recreo.")
                    cur.execute(f"drop table {tabla}")
                cur.execute(f"""
                    create table if not exists {tabla} (
                        {columnas},
                        actualizado_en timestamptz not null default now(),
                        primary key ({", ".join(t["clave"])}){chequeo}
                    )
                """)
                d = docs.get(nombre, {})
                if d.get("description"):
                    cur.execute(f"comment on table {tabla} is %s", (" ".join(d["description"].split()),))
                for col, info in d.get("columns", {}).items():
                    if info.get("description") and col in {c for c, _ in t["columnas"]}:
                        cur.execute(f'comment on column {tabla}."{col}" is %s', (info["description"],))
                cur.execute(f"comment on column {tabla}.actualizado_en is %s",
                            ("Momento en que el DAG pipeline_grupo07 publicó la fila.",))
                cur.execute(f"truncate {tabla}")
                lista = ", ".join(c for c, _ in t["columnas"])
                execute_values(cur, f"insert into {tabla} ({lista}) values %s", datos[nombre])
                cur.execute(f"select count(*), sum(cantidad_amenazas) from {tabla}")
                resultado[tabla] = cur.fetchone()
    finally:
        pg.close()
    for (tabla, (n, total)), nombre in zip(resultado.items(), TABLAS):
        if n != len(datos[nombre]):
            raise RuntimeError(f"Se esperaban {len(datos[nombre])} filas en {tabla} y hay {n}.")
        print(f"Publicadas {n} filas ({total} amenazas) en PostgreSQL: {tabla}")
    totales = {total for n, total in resultado.values()}
    if len(totales) != 1:
        raise RuntimeError(f"Las tablas gold no suman lo mismo: {resultado}")
    return sum(n for n, _ in resultado.values())


# ---------------------------------------------------------------------------- DAG
# Reintentos: para lo que puede resolverse solo (el worker se reinició, PostgreSQL o MinIO no
# respondieron un rato). Todas las tareas son idempotentes, así que reintentar nunca duplica.
# 2 reintentos con espera creciente (5 y 10 min) cubren una caída corta sin esconder un problema
# real; más que eso solo demora el aviso. Cada tarea ajusta según qué falla puede tener.
default_args = {
    "owner": "grupo07",
    "retries": 2,
    "retry_delay": pendulum.duration(minutes=5),
    "retry_exponential_backoff": True,
    "max_retry_delay": pendulum.duration(minutes=30),
    "execution_timeout": pendulum.duration(hours=1),   # una tarea colgada no bloquea el DAG
}

with DAG(
    dag_id="pipeline_grupo07",
    description="URLhaus -> bronze (Delta) -> dbt silver/gold -> PostgreSQL",
    start_date=pendulum.datetime(2024, 1, 1, tz="America/Argentina/Cordoba"),
    schedule="@daily",
    catchup=False,
    max_active_runs=1,          # dos corridas a la vez pisarían /tmp/dbt_run_grupo07
    default_args=default_args,
    tags=["proyecto-integrador", "grupo07"],
    # Reproceso de un rango en UNA corrida (Trigger DAG w/ config). Vacíos = el intervalo normal
    # de la corrida. Ver docs/BACKFILL.md.
    params={
        "desde": Param(None, type=["null", "string"],
                       description="Reproceso: primer día a reprocesar (AAAA-MM-DD, UTC). Vacío = intervalo de la corrida."),
        "hasta": Param(None, type=["null", "string"],
                       description="Reproceso: día siguiente al último a reprocesar (AAAA-MM-DD, UTC, excluido)."),
    },
) as dag:

    preparar_entorno = BashOperator(
        task_id="preparar_entorno",
        bash_command=f"""
        set -e
        if [ ! -f {VENV}/.listo_{VENV_VERSION} ]; then
            rm -rf {VENV}
            python3 -m venv {VENV}
            {VENV}/bin/pip install --quiet --upgrade pip
            {VENV}/bin/pip install --quiet {VENV_REQS}
            touch {VENV}/.listo_{VENV_VERSION}
        fi
        {VENV_PY} -c "import deltalake, duckdb, pandas, psycopg2, requests; print('venv OK')"
        test -f {ENV_FILE} || (echo "Falta {ENV_FILE}: corré la celda de despliegue de NB05" && exit 1)
        """,
    )

    comunes = dict(python=VENV_PY, expect_airflow=False)

    # Credenciales mal no se arreglan reintentando: un solo reintento, por si el servidor se
    # estaba reiniciando.
    validar_postgres = ExternalPythonOperator(
        task_id="validar_postgres", python_callable=_validar_postgres,
        op_kwargs={"env_file": ENV_FILE}, retries=1,
        retry_delay=pendulum.duration(minutes=2), **comunes)

    # El intervalo de cada corrida viaja como plantilla: Airflow la resuelve con las fechas de
    # ESA corrida (también en un backfill), y la tarea no mira el reloj para filtrar. Si se
    # dispara con los parámetros desde/hasta, reprocesa ese rango completo en una corrida.
    extraer_bronze = ExternalPythonOperator(
        task_id="extraer_bronze", python_callable=_extraer_bronze,
        op_kwargs={"env_file": ENV_FILE,
                   "inicio": "{{ params.desde or data_interval_start.isoformat() }}",
                   "fin": "{{ params.hasta or data_interval_end.isoformat() }}"},
        # Las fallas de URLhaus ya se reintentan adentro (3 intentos). Acá, un reintento más con
        # 10 min de espera, por si la caída fue más larga o se cayó el worker.
        retries=1, retry_delay=pendulum.duration(minutes=10),
        execution_timeout=pendulum.duration(hours=2),       # el reproceso baja la base completa
        **comunes)

    # Un test que falla es determinístico: volver a correrlo da lo mismo. Un solo reintento,
    # por si falló la lectura de MinIO.
    dbt_build = ExternalPythonOperator(
        task_id="dbt_build", python_callable=_dbt_build,
        op_kwargs={"env_file": ENV_FILE, "dbt_src": DBT_SRC, "dbt_run": DBT_RUN, "venv": VENV},
        retries=1, retry_delay=pendulum.duration(minutes=2), **comunes)

    publicar_gold = ExternalPythonOperator(
        task_id="publicar_gold", python_callable=_publicar_gold,
        op_kwargs={"env_file": ENV_FILE, "dbt_run": DBT_RUN}, **comunes)

    preparar_entorno >> [validar_postgres, extraer_bronze] >> dbt_build >> publicar_gold
'''
print(f"Código del DAG generado en memoria ({len(dag_code.splitlines())} líneas).")


Código del DAG generado en memoria (524 líneas).


## 3. Validar la sintaxis del DAG

Antes de desplegarlo, comprobamos que el código Python del DAG compila. Airflow no cargará un DAG con errores de sintaxis.

In [3]:
import ast
try:
    ast.parse(dag_code)
    print("✅ ¡Sintaxis perfecta! El código del DAG compila sin errores.")
except SyntaxError as e:
    print(f"❌ Error de sintaxis en la línea {e.lineno}: {e.msg}")
    raise

✅ ¡Sintaxis perfecta! El código del DAG compila sin errores.


## 3b. Configuración de dbt

NB05 también escribe `dbt_project.yml` y `profiles.yml` en `dbt/`, así el proyecto dbt queda igual en el JupyterHub (NB04) y en Airflow.

In [4]:
# Configuración de dbt que usa tanto NB04 (JupyterHub) como Airflow.
dbt_project_yml = r'''# Proyecto dbt del grupo07 — Ingeniería de Datos, LCD UCA 2026.
# Transforma bronze (URLhaus en MinIO) en silver (stg_urlhaus) y gold (fct_amenazas_mensual) sobre DuckDB.
name: 'proyecto_id'
version: '1.0.0'
config-version: 2
profile: 'ingdatos'

model-paths:  ["models"]
target-path:  "target"
clean-targets: ["target", "dbt_packages"]

# La clave tiene que ser igual a `name` (proyecto_id). Ambos modelos se materializan como tabla:
# stg_urlhaus (silver) y fct_amenazas_mensual (gold, la que se publica en PostgreSQL).
models:
  proyecto_id:
    +materialized: table
'''

profiles_yml = r'''ingdatos:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: warehouse.duckdb
      extensions:
        - httpfs
      settings:
        # Credenciales y endpoint de MinIO desde el entorno (JupyterHub o dags/.env en Airflow)
        s3_endpoint: "{{ env_var('S3_ENDPOINT', 'minio:9000') }}"
        s3_access_key_id: "{{ env_var('AWS_ACCESS_KEY_ID') }}"
        s3_secret_access_key: "{{ env_var('AWS_SECRET_ACCESS_KEY') }}"
        s3_region: "{{ env_var('AWS_REGION', 'us-east-1') }}"
        s3_use_ssl: "false"
        s3_url_style: "path"
        # date_added viene en UTC: sin esto, cast(... as timestamp) usa la zona de la máquina
        # y una URL del 1/12 00:00 UTC cae en noviembre si dbt corre con hora argentina.
        TimeZone: "UTC"
'''

dbt_dir = os.path.join(BASE, "dbt")
with open(os.path.join(dbt_dir, "dbt_project.yml"), "w", encoding="utf-8") as f:
    f.write(dbt_project_yml)
with open(os.path.join(dbt_dir, "profiles.yml"), "w", encoding="utf-8") as f:
    f.write(profiles_yml)
print("dbt_project.yml y profiles.yml escritos en", dbt_dir)

dbt_project.yml y profiles.yml escritos en /home/jovyan/work/dbt


## 3c. Modelos dbt

NB05 escribe también los modelos en `dbt/models/` y el test de cuadre en `dbt/tests/`.

* `stg_urlhaus` (silver) suma `familia_malware` y `tipo_archivo`, sacados de los **tags** de URLhaus: el campo `threat` casi siempre es `malware_download`, así que solo no alcanza para distinguir amenazas.
* `fct_familias_mensual` (gold nueva): amenazas por mes, familia y tipo de archivo, con una columna `fecha` para Metabase.
* `assert_gold_cuadra`: test que falla si la silver y las dos gold no suman lo mismo.

In [5]:
# Modelos dbt (silver y gold) + test de cuadre. Los usa NB04 y, copiados a dags/dbt, Airflow.
import shutil
stg_urlhaus_sql = r'''-- Capa silver: tipado, limpieza y deduplicado de amenazas.
-- Reglas: descartar filas sin id o sin fecha; una sola observación por ID.
-- De los tags de URLhaus se sacan la familia de malware y el tipo de archivo
-- (el campo threat casi siempre es 'malware_download', así que no alcanza para distinguir amenazas).
{{ config(materialized='table') }}

{#- Tags que indican el tipo de archivo que descarga la URL -#}
{%- set tipos_archivo = ['elf', 'exe', 'dll', 'sh', 'bash', 'bat', 'cmd', 'ps1', 'powershell',
                         'js', 'vbs', 'hta', 'lnk', 'msi', 'iso', 'img', 'zip', 'rar', '7z', 'gz',
                         'tar', 'doc', 'docx', 'xls', 'xlsx', 'pdf', 'apk', 'jar', 'py', 'php',
                         'html', 'txt', 'ascii'] -%}
{#- Tags genéricos que no son una familia: arquitectura, origen o comportamiento -#}
{%- set genericos = ['32-bit', '64-bit', 'arm', 'arm4', 'arm5', 'arm6', 'arm7', 'aarch64', 'mips',
                     'mipsel', 'mpsl', 'm68k', 'ppc', 'powerpc', 'sparc', 'sh4', 'x86', 'x86-64',
                     'x86_64', 'i586', 'i686', 'amd64', 'arc', 'opendir', 'encrypted', 'geofenced',
                     'botnetdomain', 'dropper', 'downloader', 'loader', 'payload', 'malware',
                     'malicious', 'script', 'stealer', 'rat', 'trojan', 'hfs', 'censys',
                     'unknown', 'none', 'null', 'nan',
                     'malware-download', 'c2-monitor-auto', 'dll-sideloading',
                     'cowrie', 'honeypot', 'ssh', 'telnet', 'encoded',
                     'botnet', 'iot', 'automated', 'java', 'docker', 'stego',
                     'macos', 'vscode', 'nodejs', 'windows', 'linux', 'android'] -%}
{#- Tags que son una IP (con puntos o guiones, a veces con puerto: 176-65-134-121-8080) o un
    dominio (pw-dvx.to, o con guiones en lugar de puntos: bauble-cleverpondky-com). Indican de
    dónde se descargaba el archivo, no qué malware es. -#}
{%- set patron_ip = '^[0-9]{1,3}([.-][0-9]{1,3}){3}([.:-][0-9]{1,5})?$' -%}
{%- set patron_dominio = '[.-](com|net|org|info|biz|xyz|top|site|online|space|shop|store|live|club|pw|to|cc|me|io|su|ru|cn|in|br|de|uk|us)$' -%}
{#- Tags de dónde estaba alojado el archivo (servicio o plataforma), tampoco son una familia -#}
{%- set hosting = ['github', 'gitlab', 'bitbucket', 'discord', 'dropbox', 'google-drive',
                   'googledrive', 'gdrive', 'onedrive', 'mediafire', 'mega', 'pastebin',
                   'telegram', 'cdn', 'wordpress', 'aws', 'azure', 'cloudflare', 'digitalocean',
                   'ngrok', 'ipfs', 'ftp', 'http', 'https', 'sourceforge', 'firebase'] -%}

with base as (
    select
        cast(id as integer) as id_amenaza,
        cast(date_added as timestamp) as fecha_agregado,
        cast(url as varchar) as url_maliciosa,
        cast(url_status as varchar) as estado,
        cast(threat as varchar) as tipo_amenaza,
        cast(reporter as varchar) as reportero,
        -- en bronze los tags son una lista (['Mozi', 'elf']) o texto: los pasamos a texto
        cast(tags as varchar) as etiquetas
    -- DuckDB lee directamente los archivos Parquet de tu Delta Lake en MinIO
    from read_parquet('s3://{{ env_var("MINIO_BUCKET", "grupo07") }}/bronze/series/*.parquet')
    where id is not null
      and date_added is not null
),
dedup as (
    select *,
        row_number() over (
            partition by id_amenaza
            order by fecha_agregado desc
        ) as rn
    from base
),
unicos as (
    select * from dedup where rn = 1
),
-- Una fila por tag, con su posición (el orden en que URLhaus los informa)
lista as (
    select id_amenaza,
           string_split(regexp_replace(coalesce(etiquetas, ''), '[\[\]''"]', '', 'g'), ',') as tags
    from unicos
),
tags as (
    select id_amenaza,
           trim(unnest(tags)) as tag,
           generate_subscripts(tags, 1) as pos
    from lista
),
clasificados as (
    select id_amenaza, pos,
        case when lower(tag) in ('{{ tipos_archivo | join("', '") }}') then lower(tag) end as archivo,
        case when tag = ''
               or lower(tag) in ('{{ tipos_archivo | join("', '") }}')
               or lower(tag) in ('{{ genericos | join("', '") }}')
               or lower(tag) in ('{{ hosting | join("', '") }}')
               or regexp_matches(tag, '^[A-Z]{2,3}$')        -- códigos de país: CHN, USA, ...
               or regexp_matches(tag, '{{ patron_ip }}')
               or regexp_matches(lower(tag), '{{ patron_dominio }}')
               or regexp_matches(lower(tag), '^[0-9a-f]{6,}$')   -- pedazos de hash (d52f85)
               or lower(tag) like 'dropped-by%'
               or lower(tag) like 'ua-%'
             then null else tag end as familia
    from tags
),
por_id as (
    -- el primer tag que es familia y el primero que es tipo de archivo
    select id_amenaza,
           arg_min(familia, pos) filter (where familia is not null) as familia_malware,
           arg_min(archivo, pos) filter (where archivo is not null) as tipo_archivo
    from clasificados
    group by id_amenaza
),
-- Mismo nombre con distintas mayúsculas (mirai / Mirai) = una sola familia:
-- se usa la forma más frecuente de cada nombre.
grafias as (
    select familia_malware,
           row_number() over (partition by lower(familia_malware)
                              order by count(*) desc, familia_malware) as orden
    from por_id
    where familia_malware is not null
    group by familia_malware
)
select
    u.id_amenaza,
    u.fecha_agregado,
    u.url_maliciosa,
    u.estado,
    u.tipo_amenaza,
    coalesce(g.familia_malware, 'sin_familia') as familia_malware,
    coalesce(p.tipo_archivo, 'desconocido') as tipo_archivo,
    u.etiquetas,
    u.reportero
from unicos u
left join por_id p using (id_amenaza)
left join grafias g
       on lower(g.familia_malware) = lower(p.familia_malware)
      and g.orden = 1
'''

fct_amenazas_mensual_sql = r'''-- Capa gold agregada: resumen de fct_familias_mensual por mes y tipo de amenaza.
-- Se calcula desde la gold atómica (no desde la silver), así las dos no se pueden desincronizar.

{{ config(materialized='table') }}

select
    anio,
    mes,
    tipo_amenaza,
    sum(cantidad_amenazas) as cantidad_amenazas
from {{ ref('fct_familias_mensual') }}
group by 1, 2, 3
order by anio desc, mes desc, cantidad_amenazas desc
'''

fct_familias_mensual_sql = r'''-- Capa gold atómica: el grano más fino que publicamos para el negocio / BI.
-- Grano: cantidad de amenazas por mes, tipo de amenaza, familia de malware y tipo de archivo.
-- Cada URL cuenta una sola vez. Los demás resúmenes (fct_amenazas_mensual) salen de esta tabla.

{{ config(materialized='table') }}

select
    cast(date_trunc('month', fecha_agregado) as date) as fecha,
    extract(year from fecha_agregado) as anio,
    extract(month from fecha_agregado) as mes,
    tipo_amenaza,
    familia_malware,
    tipo_archivo,
    count(id_amenaza) as cantidad_amenazas
from {{ ref('stg_urlhaus') }}
group by 1, 2, 3, 4, 5, 6
order by fecha desc, cantidad_amenazas desc
'''

schema_yml = r'''version: 2

models:
  - name: stg_urlhaus
    description: >
      Silver: una fila por URL maliciosa reportada a URLhaus (abuse.ch), sin duplicados, con los
      tipos corregidos y con la familia de malware y el tipo de archivo deducidos de las etiquetas.
      Lee directamente el bronze en MinIO (Delta Lake). Descarta filas sin id o sin fecha.
    columns:
      - name: id_amenaza
        description: "Identificador que URLhaus le asigna a la URL. Es único y crece con el tiempo: una URL nueva siempre tiene un id mayor."
        tests:
          - not_null
          - unique
      - name: fecha_agregado
        description: "Fecha y hora (UTC) en que la URL se cargó en URLhaus. No es cuándo empezó el ataque, sino cuándo alguien lo reportó."
        tests:
          - not_null
      - name: url_maliciosa
        description: "La URL que distribuye malware. Apunta a contenido activo: no abrir. Solo queda en bronze y silver; no se publica."
      - name: estado
        description: "Si la URL seguía sirviendo malware la última vez que URLhaus la revisó: online, offline o unknown."
      - name: tipo_amenaza
        description: "Clasificación de URLhaus (campo threat). Casi siempre es malware_download (el 18/11/2026 aparecieron las 2 primeras con c2, servidor de comando y control): por eso se deducen familia y tipo de archivo."
      - name: familia_malware
        description: >
          Familia de malware que distribuye la URL (Mozi, mirai, LuaJIT-loader, ...): la primera
          etiqueta que no es un tipo de archivo, una arquitectura, un hosting, un país, una IP, un
          dominio ni un término genérico. Las mayúsculas se unifican con la grafía más frecuente.
          'sin_familia' si ninguna etiqueta califica. Es aproximada: las etiquetas las carga cada
          reportero sin un formato fijo.
        tests:
          - not_null
      - name: tipo_archivo
        description: "Formato del archivo que descarga la URL según las etiquetas (elf, exe, zip, sh, ...). 'desconocido' si no hay ninguna etiqueta de tipo de archivo."
        tests:
          - not_null
      - name: etiquetas
        description: "Etiquetas originales de URLhaus, separadas por comas, tal como las cargó quien reportó la URL."
      - name: reportero
        description: "Usuario de URLhaus que reportó la URL. Identifica a una persona o equipo: no se publica en gold."

  - name: fct_amenazas_mensual
    description: >
      Gold agregada: cuántas URLs maliciosas nuevas se registraron cada mes. Se calcula sumando
      fct_familias_mensual, así que los totales de las dos tablas siempre coinciden. Responde
      "¿sube o baja el volumen de URLs maliciosas mes a mes?".
    columns:
      - name: anio
        description: "Año en que URLhaus registró las URLs."
        tests: [not_null]
      - name: mes
        description: "Mes (1 a 12) en que URLhaus registró las URLs."
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
      - name: tipo_amenaza
        description: "Clasificación de URLhaus: casi siempre malware_download; muy pocas c2."
        tests: [not_null]
      - name: cantidad_amenazas
        description: "URLs maliciosas distintas registradas en ese mes. Cada URL cuenta una sola vez."

  - name: fct_familias_mensual
    description: >
      Gold atómica: cuántas URLs maliciosas nuevas hubo cada mes, por familia de malware y por
      tipo de archivo. Es el grano más fino que se publica. Responde "¿qué familias de malware
      son las más activas y en qué formato se distribuyen, mes a mes?".
    columns:
      - name: fecha
        description: "Primer día del mes calendario (para graficar series de tiempo)."
        tests: [not_null]
      - name: anio
        description: "Año del mes."
      - name: mes
        description: "Mes (1 a 12)."
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
      - name: tipo_amenaza
        description: "Clasificación de URLhaus: casi siempre malware_download; muy pocas c2."
        tests: [not_null]
      - name: familia_malware
        description: "Familia de malware (ver stg_urlhaus). 'sin_familia' si no se pudo identificar."
        tests: [not_null]
      - name: tipo_archivo
        description: "Formato del archivo que se descarga (elf, exe, zip, ...). 'desconocido' si no se sabe."
        tests: [not_null]
      - name: cantidad_amenazas
        description: "URLs maliciosas distintas de esa familia y ese tipo de archivo registradas en el mes."

  - name: fct_features_familias
    description: >
      Features por mes y familia de malware, para anticipar qué familias crecen. Una fila por
      (mes, familia) desde el primer mes en que aparece la familia; los meses sin URLs valen 0.
      Todas las ventanas miran solo hacia atrás: el valor de un mes no usa meses posteriores.
    columns:
      - name: fecha
        description: "Primer día del mes calendario."
        tests: [not_null]
      - name: anio
        description: "Año del mes (para filtrar sin tocar la fecha)."
      - name: mes
        description: "Número de mes, 1 a 12."
      - name: familia_malware
        description: "Familia de malware (igual que en fct_familias_mensual)."
        tests: [not_null]
      - name: cantidad_amenazas
        description: "URLs nuevas de la familia registradas en el mes (0 si no hubo)."
        tests: [not_null]
      - name: cantidad_mes_anterior
        description: "URLs de la familia en el mes calendario anterior (0 si no hubo o si es su primer mes)."
        tests: [not_null]
      - name: variacion_mensual_pct
        description: "Cambio % contra el mes anterior. Vacío si el mes anterior tuvo 0 URLs (no se puede dividir)."
      - name: promedio_movil_3m
        description: "Promedio de URLs por mes en el mes actual y los dos anteriores (menos meses si la familia es más nueva)."
        tests: [not_null]
      - name: acumulado
        description: "Total de URLs de la familia desde su primera aparición hasta este mes inclusive."
        tests: [not_null]
      - name: pct_del_mes
        description: "Porcentaje que representa la familia sobre todas las URLs nuevas del mes."
        tests: [not_null]
      - name: mes_completo
        description: "Falso solo para el último mes con datos, que puede estar incompleto: sus valores van a cambiar."
        tests: [not_null]
'''

assert_gold_cuadra_sql = r'''-- Las tablas gold, la de features y la silver tienen que sumar lo mismo: cada URL cuenta una sola vez.
-- El test falla si esta consulta devuelve alguna fila.
with totales as (
    select
        (select count(*) from {{ ref('stg_urlhaus') }}) as silver,
        (select sum(cantidad_amenazas) from {{ ref('fct_amenazas_mensual') }}) as gold_tipo,
        (select sum(cantidad_amenazas) from {{ ref('fct_familias_mensual') }}) as gold_familia,
        (select sum(cantidad_amenazas) from {{ ref('fct_features_familias') }}) as features
)
select * from totales
where silver <> gold_tipo or silver <> gold_familia or silver <> features
'''

# Tests propios de dominio (además de los genéricos de schema.yml)
assert_fechas_validas_sql = r'''-- Regla de dominio: URLhaus empezó a registrar URLs en 2018 y no puede haber URLs registradas
-- en el futuro. Una fecha fuera de ese rango indica un error de parseo o de zona horaria en la
-- ingesta (por ejemplo, una fecha local tomada como UTC o un campo mal leído del CSV).
-- El test falla si esta consulta devuelve alguna fila.
select id_amenaza, fecha_agregado
from {{ ref('stg_urlhaus') }}
where fecha_agregado < timestamp '2018-01-01'
   or fecha_agregado > cast(now() as timestamp) + interval 1 day
'''

assert_familia_no_es_tag_generico_sql = r'''-- Regla de dominio: una familia de malware nunca puede ser un tag genérico de URLhaus
-- (arquitectura, tipo de archivo, hosting, país, IP o dominio). Si aparece uno, la lista de tags genéricos
-- de stg_urlhaus quedó desactualizada y el ranking de familias del tablero estaría mal.
-- El test falla si esta consulta devuelve alguna fila.
select familia_malware, count(*) as urls
from {{ ref('stg_urlhaus') }}
where lower(familia_malware) in ('elf', 'exe', 'zip', 'sh', '32-bit', '64-bit', 'arm', 'mips',
                                 'x86', 'github', 'discord', 'dropbox', 'opendir', 'ua-wget',
                                 'malware-download', 'c2-monitor-auto', 'dll-sideloading',
                                 'cowrie', 'ssh', 'encoded', 'botnet', 'iot', 'docker',
                                 'macos', 'vscode', 'nodejs')
   or regexp_matches(lower(familia_malware), '^[0-9a-f]{6,}$')
   or regexp_matches(familia_malware, '^[A-Z]{2,3}$')
   -- una IP (176-65-139-204, 94.154.43.26, con o sin puerto)
   or regexp_matches(familia_malware, '^[0-9]{1,3}([.-][0-9]{1,3}){3}([.:-][0-9]{1,5})?$')
   -- un dominio (pw-dvx.to, bauble-cleverpondky-com)
   or regexp_matches(lower(familia_malware), '[.-](com|net|org|xyz|top|site|space|to|ru|cn)$')
group by familia_malware
'''

assert_zona_horaria_utc_sql = r'''-- URLhaus publica date_added en UTC. Si DuckDB corre con otra zona horaria (por ejemplo la
-- de una máquina en Argentina), cast(... as timestamp) corre las fechas 3 horas y las URLs de
-- las últimas horas de cada mes se cuentan en el mes anterior. profiles.yml fija TimeZone=UTC.
-- El test falla si esta consulta devuelve alguna fila.
select current_setting('TimeZone') as zona_horaria
where current_setting('TimeZone') <> 'UTC'
'''

# Features (E4): definidos una sola vez, en dbt, con tests de fuga temporal y de consistencia
fct_features_familias_sql = r'''-- Features por familia de malware y mes: variables derivadas que un modelo podría usar para
-- anticipar qué familias van a crecer. Definidas UNA sola vez, acá; el DAG las publica en
-- PostgreSQL y Metabase las lee de ahí.
--
-- Grano: una fila por (mes, familia), desde el primer mes en que aparece la familia hasta el
-- último mes con datos. Los meses sin URLs de esa familia se completan con 0: así "el mes
-- anterior" y "los últimos 3 meses" son siempre meses calendario, aunque la familia no haya
-- tenido actividad.
--
-- Sin fuga temporal: todas las ventanas miran solo hacia atrás (lag, "rows between 2 preceding
-- and current row", "unbounded preceding"). El feature de un mes nunca usa meses posteriores.
-- Lo verifica el test assert_features_sin_fuga.

{{ config(materialized='table') }}

with mensual as (
    select fecha, familia_malware, sum(cantidad_amenazas) as cantidad
    from {{ ref('fct_familias_mensual') }}
    group by 1, 2
),
limites as (
    select min(fecha) as primero, max(fecha) as ultimo from mensual
),
meses as (
    select cast(m as date) as fecha
    from limites, range(primero, ultimo + interval 1 month, interval 1 month) as t(m)
),
familias as (
    select familia_malware, min(fecha) as desde from mensual group by 1
),
grilla as (
    select m.fecha, f.familia_malware
    from familias f
    join meses m on m.fecha >= f.desde
),
base as (
    select g.fecha, g.familia_malware, coalesce(m.cantidad, 0) as cantidad_amenazas
    from grilla g
    left join mensual m using (fecha, familia_malware)
),
con_ventanas as (
    select
        b.*,
        coalesce(lag(cantidad_amenazas) over w, 0)                              as cantidad_mes_anterior,
        avg(cantidad_amenazas) over (w rows between 2 preceding and current row) as promedio_movil_3m,
        sum(cantidad_amenazas) over (w rows between unbounded preceding and current row) as acumulado,
        sum(cantidad_amenazas) over (partition by fecha)                        as total_mes
    from base b
    window w as (partition by familia_malware order by fecha)
)
select
    fecha,
    cast(extract(year from fecha) as integer)   as anio,
    cast(extract(month from fecha) as integer)  as mes,
    familia_malware,
    cast(cantidad_amenazas as bigint)           as cantidad_amenazas,
    cast(cantidad_mes_anterior as bigint)       as cantidad_mes_anterior,
    case when cantidad_mes_anterior > 0
         then round(100.0 * (cantidad_amenazas - cantidad_mes_anterior) / cantidad_mes_anterior, 2)
    end                                         as variacion_mensual_pct,
    round(promedio_movil_3m, 2)                 as promedio_movil_3m,
    cast(acumulado as bigint)                   as acumulado,
    -- Meses sin ninguna URL (huecos del histórico): 0 / 0 daría NaN, que DuckDB no trata como nulo
    case when total_mes > 0 then round(100.0 * cantidad_amenazas / total_mes, 2) else 0 end as pct_del_mes,
    fecha < (select ultimo from limites)        as mes_completo
from con_ventanas
'''

assert_features_sin_fuga_sql = r'''-- Fuga temporal: cada feature de un mes tiene que poder calcularse SOLO con datos de ese mes o
-- anteriores. Este test los recalcula de cero desde la gold atómica, filtrando explícitamente
-- fecha <= mes del feature, y compara. Si alguien cambia una ventana para que mire hacia
-- adelante (por ejemplo "following" o el promedio del mes completo), los valores no coinciden.
-- El test falla si esta consulta devuelve alguna fila.
with gold as (
    select fecha, familia_malware, sum(cantidad_amenazas) as cantidad
    from {{ ref('fct_familias_mensual') }}
    group by 1, 2
),
desde as (
    select familia_malware, min(fecha) as primero from gold group by 1
),
recalculado as (
    select
        f.fecha,
        f.familia_malware,
        coalesce((select sum(g.cantidad) from gold g
                  where g.familia_malware = f.familia_malware
                    and g.fecha = f.fecha - interval 1 month), 0)            as anterior,
        coalesce((select sum(g.cantidad) from gold g
                  where g.familia_malware = f.familia_malware
                    and g.fecha <= f.fecha), 0)                              as acumulado,
        coalesce((select sum(g.cantidad) from gold g
                  where g.familia_malware = f.familia_malware
                    and g.fecha <= f.fecha
                    and g.fecha >= f.fecha - interval 2 month), 0)
          / least(3, date_diff('month', d.primero, f.fecha) + 1)             as promedio
    from {{ ref('fct_features_familias') }} f
    join desde d using (familia_malware)
)
select f.fecha, f.familia_malware,
       f.cantidad_mes_anterior, r.anterior,
       f.acumulado, r.acumulado as acumulado_recalculado,
       f.promedio_movil_3m, round(r.promedio, 2) as promedio_recalculado
from {{ ref('fct_features_familias') }} f
join recalculado r using (fecha, familia_malware)
where f.cantidad_mes_anterior <> r.anterior
   or f.acumulado <> r.acumulado
   or abs(f.promedio_movil_3m - r.promedio) > 0.01
'''

assert_features_consistentes_sql = r'''-- Reglas que tiene que cumplir cualquier fila de features:
--   * una sola fila por (mes, familia),
--   * cantidades y acumulados no negativos, y el acumulado nunca menor que el mes,
--   * participación en el mes entre 0 % y 100 %,
--   * solo el último mes puede figurar como incompleto.
-- El test falla si esta consulta devuelve alguna fila.
with duplicados as (
    select fecha, familia_malware, 'grano duplicado' as problema
    from {{ ref('fct_features_familias') }}
    group by 1, 2
    having count(*) > 1
),
fuera_de_rango as (
    select fecha, familia_malware, 'valor fuera de rango' as problema
    from {{ ref('fct_features_familias') }}
    where cantidad_amenazas < 0
       or cantidad_mes_anterior < 0
       or acumulado < cantidad_amenazas
       or pct_del_mes < 0 or pct_del_mes > 100
),
incompletos as (
    select fecha, familia_malware, 'mes incompleto que no es el último' as problema
    from {{ ref('fct_features_familias') }}
    where not mes_completo
      and fecha < (select max(fecha) from {{ ref('fct_features_familias') }})
)
select * from duplicados
union all select * from fuera_de_rango
union all select * from incompletos
'''

archivos = {
    "models/stg_urlhaus.sql": stg_urlhaus_sql,
    "models/fct_amenazas_mensual.sql": fct_amenazas_mensual_sql,
    "models/fct_familias_mensual.sql": fct_familias_mensual_sql,
    "models/fct_features_familias.sql": fct_features_familias_sql,
    "models/schema.yml": schema_yml,
    "tests/assert_gold_cuadra.sql": assert_gold_cuadra_sql,
    "tests/assert_fechas_validas.sql": assert_fechas_validas_sql,
    "tests/assert_familia_no_es_tag_generico.sql": assert_familia_no_es_tag_generico_sql,
    "tests/assert_zona_horaria_utc.sql": assert_zona_horaria_utc_sql,
    "tests/assert_features_sin_fuga.sql": assert_features_sin_fuga_sql,
    "tests/assert_features_consistentes.sql": assert_features_consistentes_sql,
}
dbt_dir = os.path.join(BASE, "dbt")
for ruta, contenido in archivos.items():
    destino = os.path.join(dbt_dir, ruta)
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    with open(destino, "w", encoding="utf-8") as f:
        f.write(contenido)
    print("✅ escrito", destino)

# Jupyter crea .ipynb_checkpoints al abrir un .sql/.yml y dbt los toma como modelos: los borramos
for sub in ("models", "tests"):
    shutil.rmtree(os.path.join(dbt_dir, sub, ".ipynb_checkpoints"), ignore_errors=True)


✅ escrito /home/jovyan/work/dbt/models/stg_urlhaus.sql
✅ escrito /home/jovyan/work/dbt/models/fct_amenazas_mensual.sql
✅ escrito /home/jovyan/work/dbt/models/fct_familias_mensual.sql
✅ escrito /home/jovyan/work/dbt/models/fct_features_familias.sql
✅ escrito /home/jovyan/work/dbt/models/schema.yml
✅ escrito /home/jovyan/work/dbt/tests/assert_gold_cuadra.sql
✅ escrito /home/jovyan/work/dbt/tests/assert_fechas_validas.sql
✅ escrito /home/jovyan/work/dbt/tests/assert_familia_no_es_tag_generico.sql
✅ escrito /home/jovyan/work/dbt/tests/assert_zona_horaria_utc.sql
✅ escrito /home/jovyan/work/dbt/tests/assert_features_sin_fuga.sql
✅ escrito /home/jovyan/work/dbt/tests/assert_features_consistentes.sql


In [6]:
# Documentación para el repositorio: governance (6 secciones) y diseño del backfill
governance_md = r'''# Governance del dataset — Amenazas URLhaus (grupo07)

Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07

## 1. Identificación

| | |
| --- | --- |
| **Nombre** | Amenazas URLhaus mensuales |
| **Qué contiene** | Cuántas URLs nuevas que distribuyen malware se reportaron cada mes a URLhaus, separadas por familia de malware (Mozi, mirai, LuaJIT-loader, ...) y por el tipo de archivo que descargan (ejecutable de Linux, de Windows, comprimido, ...). Incluye además variables derivadas para seguir la tendencia de cada familia. |
| **Para qué existe** | Para responder qué familias de malware están más activas, cómo cambian de un mes a otro y en qué formato se distribuyen, y así priorizar reglas de bloqueo y detección. |
| **Responsable** | **Francisco Lauria** (grupo07). Si un número está mal, abrir un *issue* en el repositorio o escribirle directamente. |
| **Dónde está** | PostgreSQL, base `warehouse`, esquema `grupo07`: `fct_familias_mensual`, `fct_amenazas_mensual` y `fct_features_familias`. Se consulta en el tablero "Amenazas URLhaus — grupo07" de Metabase. |
| **Actualización** | Todos los días a las 00:00 (hora de Córdoba), con el DAG `pipeline_grupo07` de Airflow. |

## 2. Origen y linaje

**Origen.** [URLhaus](https://urlhaus.abuse.ch) es un proyecto de abuse.ch (abuse.ch AG, Suiza), que opera en cooperación con Spamhaus. Investigadores y equipos de seguridad de todo el mundo reportan ahí URLs que están distribuyendo malware. Se actualiza en forma continua: de cientos a miles de URLs nuevas por día según el período (unas 500 por día en agosto y septiembre de 2026, casi 4.000 en el pico de octubre). Se accede con una Auth-Key gratuita.

Usamos dos accesos de URLhaus:
- La **API** `/v1/urls/recent/` (las URLs de los últimos 3 días, máximo 1000), para la ingesta incremental de NB02.
- Los **exports CSV** (`recent.csv`, los últimos 30 días, y `full.csv.zip`), para la ingesta diaria del DAG y para reprocesar el pasado.

**Histórico y cobertura.** Hay datos desde junio de 2018, pero el histórico es **parcial**: el export de URLhaus conserva bien solo los últimos uno o dos meses (para septiembre de 2026 trajo prácticamente las mismas URLs que ya teníamos; para julio, 124 contra 4.580). Lo que acumulamos día a día no se pierde, pero lo que el origen ya no publica no se puede recuperar (ver *Huecos* en la sección 5).

**Recorrido de los datos.**

1. **Bronze** (Delta Lake en MinIO, `s3://grupo07/bronze/series`): una fila por URL, tal como la publica URLhaus. Nunca se borra: cada carga agrega las URLs nuevas y actualiza las que ya estaban (por ejemplo, si pasaron de online a offline).
2. **Silver** (`stg_urlhaus`): una fila por URL sin duplicados, con fechas en UTC, y con la familia de malware y el tipo de archivo deducidos de las etiquetas, descartando las que no son una familia (arquitecturas, servicios de hosting, países, IPs, dominios, honeypots, términos genéricos).
3. **Gold** (`fct_familias_mensual`): cuenta las URLs por mes, familia y tipo de archivo. `fct_amenazas_mensual` es su total por mes.
4. **Features** (`fct_features_familias`): por mes y familia, el mes anterior, la variación, el promedio de 3 meses, el acumulado y la participación en el mes, usando siempre solo datos de ese mes o anteriores.
5. **Publicación**: las tres tablas se copian a PostgreSQL en una sola operación (si algo falla, queda la versión anterior completa) y de ahí las lee Metabase.

**Diagrama de linaje**, generado de los `ref()` de los modelos dbt con `scripts/linaje.py`, no dibujado a mano (el grafo interactivo sale de `dbt docs generate --static`, sección 4 de NB04):

![Linaje de los modelos dbt](linaje.png)

## 3. Diccionario de datos

### `fct_familias_mensual` — URLs nuevas por mes, familia y tipo de archivo

| Columna | Tipo | Significado | Ejemplo |
| --- | --- | --- | --- |
| `fecha` | date | Primer día del mes en que URLhaus registró las URLs | `2026-10-01` |
| `anio` | integer | Año de ese mes | `2026` |
| `mes` | integer | Número de mes, 1 a 12 | `10` |
| `tipo_amenaza` | text | Clasificación de URLhaus. Casi siempre `malware_download` (la URL descarga malware); el 18/11/2026 aparecieron las 2 primeras `c2` (servidor de comando y control). URLhaus no documenta qué otros valores puede tomar | `malware_download` |
| `familia_malware` | text | Familia de malware que distribuyen las URLs. Sale de las etiquetas que carga quien reporta, así que es aproximada. `sin_familia` si ninguna etiqueta la identifica | `LuaJIT-loader` |
| `tipo_archivo` | text | Formato del archivo que se descarga: `elf` (ejecutable de Linux, típico de dispositivos IoT y routers), `exe` (Windows), `zip`, `sh` (script de Linux), etc. `desconocido` si no hay etiqueta de tipo de archivo | `zip` |
| `cantidad_amenazas` | bigint | URLs maliciosas distintas registradas ese mes con esa familia y ese tipo de archivo. Cada URL cuenta una sola vez | `13755` |
| `actualizado_en` | timestamptz | Cuándo el pipeline publicó la fila | `2026-11-18 20:12:56+00` |

### `fct_amenazas_mensual` — URLs nuevas por mes

Las mismas `anio`, `mes`, `tipo_amenaza`, `cantidad_amenazas` y `actualizado_en`, sumadas por mes. Coincide siempre con la suma de `fct_familias_mensual` (se calcula a partir de ella).

### `fct_features_familias` — Tendencia de cada familia

Una fila por mes y familia, desde el primer mes en que aparece la familia. Los meses en que la familia no tuvo URLs aparecen con 0.

| Columna | Tipo | Significado | Ejemplo |
| --- | --- | --- | --- |
| `fecha`, `anio`, `mes` | date, integer | El mes, como arriba | `2026-10-01` |
| `familia_malware` | text | La familia | `SalatStealer` |
| `cantidad_amenazas` | bigint | URLs nuevas de la familia en el mes | `7820` |
| `cantidad_mes_anterior` | bigint | URLs de la familia en el mes calendario anterior (0 si no tuvo o si es su primer mes) | `2` |
| `variacion_mensual_pct` | numeric | Cambio % contra el mes anterior. Vacío cuando el mes anterior tuvo 0 URLs: no se puede calcular un porcentaje sobre 0 | `390900.00` |
| `promedio_movil_3m` | numeric | Promedio de URLs por mes entre este mes y los dos anteriores (menos meses si la familia es más nueva) | `2608.67` |
| `acumulado` | bigint | Total de URLs de la familia desde su primera aparición hasta este mes | `7840` |
| `pct_del_mes` | numeric | Qué porcentaje de todas las URLs nuevas del mes son de esta familia (0 en meses sin ninguna URL) | `17.46` |
| `mes_completo` | boolean | Falso solo para el último mes, que todavía está en curso: sus valores van a cambiar | `true` |

Ningún valor de una fila usa información de meses posteriores (lo verifica un test que los recalcula desde cero).

**Campos que no terminamos de entender:**
- `tipo_amenaza`: URLhaus no documenta qué valores existen además de `malware_download`; el 18/11/2026 aparecieron 2 URLs con `c2`. Lo publicamos tal cual, así que un mes puede tener más de una fila en `fct_amenazas_mensual`.
- El estado de una URL (`online`, `offline`, `unknown`, solo en silver): no está documentado cada cuánto URLhaus lo revisa. Lo interpretamos como "el último estado que vio URLhaus".

## 4. Clasificación de sensibilidad

Tres niveles: **Público** (ya es público en el origen), **Interno** (no es secreto, pero no se publica fuera del equipo), **Sensible** (identifica o describe a una persona).

| Campo | Dónde está | Nivel | Por qué |
| --- | --- | --- | --- |
| Todas las columnas de las tres tablas publicadas | gold, PostgreSQL, Metabase | **Público** | Son conteos agregados por mes, familia y tipo de archivo. Ninguna fila corresponde a una URL, a un sitio ni a una persona. |
| `id_amenaza`, `fecha_agregado`, `tipo_amenaza`, `estado`, `familia_malware`, `tipo_archivo`, `etiquetas` | bronze y silver | **Público** | URLhaus los muestra en su sitio para cada URL. |
| `url_maliciosa` | bronze y silver | **Interno** | Es pública en URLhaus, pero apunta a malware que puede seguir activo. Además, muchas son sitios legítimos que fueron hackeados, y la URL identifica al sitio afectado (y a veces a su dueño). No se publica ni se muestra como enlace. |
| `reportero` | bronze y silver | **Interno, con criterio de sensible** | Es el usuario de URLhaus que reportó la URL: identifica a una persona o a un equipo. Es público en URLhaus, pero no lo necesitamos para la pregunta del dataset, así que no sale de silver (minimización). |

**¿La combinación de campos podría identificar a alguien?** En las tablas publicadas, no: no tienen URL, reportero ni sitio, y el grano más fino es mes × familia × tipo de archivo. El caso límite es una familia con una sola URL en un mes. Ese conteo no dice qué URL es ni quién la reportó, pero cruzándolo con el sitio público de URLhaus se podría encontrar. Lo aceptamos porque la información ya es pública en el origen y no agrega nada sobre una persona.

**Credenciales:** la Auth-Key de URLhaus y las contraseñas de MinIO y PostgreSQL viven en variables de entorno y en `dags/.env`, que no se versiona (el CI falla si aparece un `.env` o una clave en el repositorio). No figuran en ningún notebook ni en los logs de Airflow.

## 5. Calidad y garantías

**Qué se verifica en cada corrida.** Si alguna de estas verificaciones falla, no se publica nada y PostgreSQL conserva la última versión buena.
- No hay URLs duplicadas ni sin identificador. Todas tienen fecha, familia y tipo de archivo (aunque sean `sin_familia` o `desconocido`).
- Las fechas están entre 2018 (inicio de URLhaus) y hoy. El pipeline trabaja en UTC.
- Ninguna "familia" es en realidad una arquitectura, un tipo de archivo, un servicio de hosting, un país, una IP, un dominio o un término genérico.
- Silver, las dos gold y los features suman exactamente la misma cantidad de URLs.
- Ningún feature usa meses posteriores. Las participaciones están entre 0 % y 100 %, y no hay mes y familia repetidos.
- En PostgreSQL, claves primarias y meses entre 1 y 12. Además, las tres tablas tienen que sumar lo mismo para que la publicación se confirme.

**Frescura.** Se actualiza una vez por día, a las 00:00 de Córdoba (03:00 UTC), con las URLs del día anterior. Una URL reportada hoy aparece mañana. `actualizado_en` dice cuándo fue la última publicación.

**Huecos conocidos.**
- **El histórico anterior a agosto de 2026 está incompleto.** Los meses de 2018 a mayo de 2026 tienen entre unas pocas y unos cientos de URLs, y junio y julio unas 150 por día: son las que el origen todavía conserva, más las que fuimos acumulando. Agosto y septiembre (unas 500 por día) probablemente estén cerca de completos. No hay una fuente independiente para medir cuánto falta.
- **Octubre de 2026 fue un pico de reportes, no un efecto de cobertura:** casi 4.000 URLs por día, ocho veces agosto y septiembre; noviembre vuelve a unas 1.100 por día. Coincide con un solo reportero, `wristhulk`, que desde el 29/09/2026 cargó 96.956 URLs, el 51 % de todo el dataset; en el mismo período aparecen en masa unas pocas familias (SmartLoader, SalatStealer, zapret, LuaJIT-loader). Con estos datos no se puede separar cuánto es más malware y cuánto es más gente reportando.
- **El mes en curso está incompleto** (`mes_completo = false`).
- **18.362 URLs, recuperadas el 18/11/2026 después de un reproceso que las había borrado** (ver `INCIDENTE_2026-11-18_backfill.md`), no tienen la fecha en que se las vio activas por última vez. Ese dato está solo en bronze y no se usa en gold.
- Algunos meses viejos no tienen ninguna URL (por ejemplo, 2019-09 o 2021-02).

**Lo que este dataset no permite responder.**
- **El volumen real de amenazas antes de agosto de 2026,** ni tendencias de largo plazo: la cobertura cambia con el tiempo.
- **Cuándo empezó un ataque:** la fecha es la del reporte, no la del ataque.
- **El malware que nadie reportó a URLhaus.** El dataset refleja lo que reporta su comunidad, y está muy concentrado: un solo reportero cargó la mitad de todas las URLs. Un pico puede ser un reportero muy activo y no más malware.
- **A quién atacan:** no hay datos de víctimas, de países afectados ni de sectores.
- **Rankings exactos de familias:** la familia es aproximada, porque depende de las etiquetas que pone cada reportero. Según el mes, entre el 10 % y un tercio de las URLs no tiene ninguna familia identificable (`sin_familia`).

## 6. Licencia y uso

**Licencia del origen.** Los [Términos de Uso de abuse.ch](https://abuse.ch/terms-of-use/) (vigentes desde el 4/11/2025, última actualización el 21/10/2026) establecen:
- URLhaus es de acceso gratuito para usuarios autenticados y **con fines no comerciales**, con límites de uso.
- El uso comercial puede requerir una suscripción paga, gestionada por Spamhaus.
- No se pueden hacer obras derivadas de las plataformas ni usar los datos para entrenar IA sin consentimiento.
- **URLhaus no tiene licencia CC0.** CC0 aplica a otros proyectos de abuse.ch (Feodo Tracker, SSL Blacklist, YARAify).
- abuse.ch puede restringir el acceso si no se respetan estas condiciones.

**Qué se puede hacer con este dataset.**
- Usarlo con fines académicos y no comerciales, citando a URLhaus (abuse.ch) como fuente.

**Qué no se puede hacer.**
- Usarlo con fines comerciales.
- Redistribuirlo como producto.
- Usarlo para entrenar modelos de IA sin autorización de abuse.ch y Spamhaus.

**¿Nuestros datos derivados tienen la misma licencia que el origen?** No lo tenemos resuelto. Las tablas publicadas son conteos agregados, no copias de los registros, y podría argumentarse que son un trabajo nuevo. Pero los términos prohíben las "obras derivadas" de las plataformas sin aclarar si un agregado estadístico cuenta como tal. Por eso los tratamos con las mismas restricciones que el origen: uso académico, sin fines comerciales y citando la fuente.

**Código del pipeline:** desarrollado por grupo07 para la materia Ingeniería de Datos (LCD, UCA Rosario, 2026), con fines académicos.
'''

backfill_md = r'''# Backfill del DAG `pipeline_grupo07`

Cada corrida procesa **solo su intervalo** `[data_interval_start, data_interval_end)`, el que le asigna Airflow. Ninguna tarea usa la fecha de hoy para decidir qué filas procesar, así que un backfill de enero de 2025 trae las URLs de enero de 2025.

## Por qué no usamos `/v1/urls/recent/`

El endpoint `/v1/urls/recent/` de la API de URLhaus no acepta fechas: siempre devuelve lo registrado en los últimos 3 días, con un máximo de 1000 URLs. Con él, un backfill traería las amenazas de hoy. Por eso `extraer_bronze` usa los **volcados CSV** que publica URLhaus, con las mismas columnas que la API:

| Intervalo pedido | Fuente |
| --- | --- |
| Dentro de los últimos 30 días | `https://urlhaus-api.abuse.ch/v2/files/exports/<AUTH-KEY>/recent.csv` |
| Más viejo | `https://urlhaus-api.abuse.ch/v2/files/exports/<AUTH-KEY>/full.csv.zip` (base completa) |

La Auth-Key va dentro de la URL, como lo pide abuse.ch, así que el DAG la reemplaza por `<AUTH-KEY>` en los logs y en los mensajes de error. Las direcciones viejas (`urlhaus.abuse.ch/downloads/...`) responden 410 Gone.

El reloj solo se usa para elegir cuál de los dos volcados descargar, porque el chico alcanza para lo reciente. Las filas siempre se filtran por el intervalo de Airflow. El volcado se guarda en `/tmp` durante 6 horas, así que un backfill de muchos días lo descarga una sola vez.

## Idempotencia

1. `extraer_bronze` toma del CSV las URLs con `dateadded` en el intervalo y hace **upsert por `id`** en bronze (`merge`: la URL que ya estaba se actualiza, la nueva se agrega, **ninguna se borra**). Correr dos veces el mismo intervalo deja el mismo resultado, sin duplicados.

   Antes se reemplazaba el intervalo (`overwrite` con `predicate`). Eso supone que el export trae todas las URLs de esas fechas, y no es así: el 18/11/2026 un reproceso de 2018 a hoy dejó a julio de 2026 con 124 URLs en lugar de 4.580. Ver [`INCIDENTE_2026-11-18_backfill.md`](INCIDENTE_2026-11-18_backfill.md).
2. `dbt_build` recalcula silver y gold desde todo bronze.
3. `publicar_gold` reemplaza las tablas de PostgreSQL en una transacción.

`max_active_runs=1` hace que las corridas de un backfill vayan de a una.

## Cómo correrlo

No tenemos la CLI de Airflow en el servidor de la cátedra: todo se hace desde la interfaz, con ▶ → **Trigger DAG w/ config**.

**Un día puntual.** Poner la *Logical date* del día (por ejemplo `2025-03-15`) y dejar `desde` y `hasta` vacíos. La corrida procesa `[data_interval_start, data_interval_end)` de esa fecha. El log de `extraer_bronze` muestra `Intervalo a procesar: [...]` y cuántas URLs había.

**Qué día procesa.** Una corrida procesa el intervalo que **termina** en su *Logical date*: la corrida de las 00:00 del 10/11 procesa todo el 9/11 (hora de Córdoba), el día que acaba de cerrar. Ensayado el 18/11/2026: *Logical date* `2026-11-10` → `Intervalo a procesar: [2026-11-09 03:00 UTC, 2026-11-10 03:00 UTC)`, 914 URLs. Para reprocesar el día X: *Logical date* X + 1, o `desde = X` y `hasta = X + 1`.

**Un rango (meses o años).** Completar los parámetros `desde` (primer día, `AAAA-MM-DD`) y `hasta` (día siguiente al último, excluido). Por ejemplo `desde = 2018-01-01`, `hasta = 2026-11-01` reprocesa todo el histórico hasta octubre de 2026 **en una sola corrida**. El log lista cuántas URLs escribió por mes.

## ¿Qué pasa si hay que reprocesar dos años?

Con el backfill clásico de Airflow serían unas 730 corridas diarias. Cada una descargaría la base completa de URLhaus (la caché de 6 horas ayuda, pero un backfill largo la vence), correría `dbt build` entero y republicaría la gold: horas de trabajo repetido para el mismo resultado, y además sin CLI habría que dispararlas una por una.

Por eso el DAG acepta un **rango en una sola corrida**:

1. Descarga `full.csv.zip` una vez.
2. Lo lee en streaming, agrupa las URLs por mes y hace el mismo upsert por `id` que una corrida diaria. Nunca tiene en memoria más de 300.000 filas: si se llena, vuelca a bronze lo que tiene y sigue.
3. Después corre `dbt build` y `publicar_gold` **una vez**, para todo el rango.

Es idempotente y no pierde datos: correr dos veces el mismo rango deja bronze igual, una corrida diaria encima no duplica, y si el export dejó de traer URLs que ya teníamos, bronze las conserva (probado con un histórico simulado de 7.550 URLs, mezcladas, volcando cada 1.000 filas y con un segundo export al que le faltan 775 URLs).

**Límites:** el export `full.csv.zip` de URLhaus **no es un histórico completo**: trae casi todo el último mes y, de los meses anteriores, solo una parte. Un reproceso no puede recuperar lo que el origen ya no publica; solo completa lo que todavía está. Lo que se acumuló en bronze día a día es la mejor fuente para el pasado, y por eso bronze nunca borra.
'''

docs_dir = os.path.join(BASE, "docs")
os.makedirs(docs_dir, exist_ok=True)
for nombre, contenido in {"GOVERNANCE.md": governance_md, "BACKFILL.md": backfill_md}.items():
    with open(os.path.join(docs_dir, nombre), "w", encoding="utf-8") as f:
        f.write(contenido)
    print("✅ escrito", os.path.join(docs_dir, nombre))


✅ escrito /home/jovyan/work/docs/GOVERNANCE.md
✅ escrito /home/jovyan/work/docs/BACKFILL.md


## 4. Desplegar

Escribimos el DAG a `dags/` y dejamos ahí todo lo que Airflow necesita. Airflow solo ve esa carpeta (`/opt/airflow/dags/grupo07/`): no ve el resto de `/home/jovyan/work`. Después de unos 5 minutos, activá el DAG en Airflow y disparalo manualmente (*Trigger*).

In [7]:
# Escribir el DAG en dags/ (la carpeta que Airflow ve como /opt/airflow/dags/grupo07/)
os.makedirs(DAGS_DIR, exist_ok=True)
destino = os.path.join(DAGS_DIR, "pipeline_grupo07.py")
with open(destino, "w", encoding="utf-8") as f:
    f.write(dag_code)
print("✅ DAG escrito en:", destino)

✅ DAG escrito en: /home/jovyan/work/dags/pipeline_grupo07.py


In [8]:
# Dejar dentro de dags/ todo lo que Airflow necesita: dbt, credenciales y .airflowignore
import ast, shutil, getpass

# 1) Validar la sintaxis del DAG
dag = os.path.join(DAGS_DIR, "pipeline_grupo07.py")
ast.parse(open(dag, encoding="utf-8").read())
print("1) DAG OK:", dag)

# 2) Que Airflow no lea copias de Jupyter, dbt ni scripts viejos
shutil.rmtree(os.path.join(DAGS_DIR, ".ipynb_checkpoints"), ignore_errors=True)
with open(os.path.join(DAGS_DIR, ".airflowignore"), "w") as f:
    f.write(".ipynb_checkpoints\ndbt\nscripts\n__pycache__\n")
print("2) .airflowignore escrito")

# 3) Copiar el proyecto dbt (work/dbt es el original, el que usa NB04)
shutil.copytree(os.path.join(BASE, "dbt"), os.path.join(DAGS_DIR, "dbt"), dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("target", "logs", ".ipynb_checkpoints",
                                              "*.duckdb", "*.duckdb.wal"))
print("3) dbt copiado a", os.path.join(DAGS_DIR, "dbt"))

# 4) Credenciales para Airflow: variables del JupyterHub + API key de URLhaus.
#    La API key se reutiliza del .env anterior o se pide por teclado: así no queda escrita
#    en el notebook ni en el repositorio.
ruta_env = os.path.join(DAGS_DIR, ".env")
anterior = {}
if os.path.exists(ruta_env):
    for linea in open(ruta_env, encoding="utf-8"):
        if "=" in linea:
            k, v = linea.strip().split("=", 1)
            anterior[k] = v
clave = (os.environ.get("URLHAUS_AUTH_KEY") or anterior.get("URLHAUS_AUTH_KEY")
         or getpass.getpass("API key de URLhaus: "))

claves = ["MINIO_BUCKET", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_ENDPOINT_URL",
          "PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]
lineas = [f"{k}={os.environ[k]}" for k in claves if os.environ.get(k)]
lineas.append(f"URLHAUS_AUTH_KEY={clave}")
esquema = os.environ.get("PG_SCHEMA") or anterior.get("PG_SCHEMA")   # opcional
if esquema:
    lineas.append(f"PG_SCHEMA={esquema}")
with open(ruta_env, "w", encoding="utf-8") as f:
    f.write("\n".join(lineas) + "\n")
print("4) .env escrito con:", [l.split("=")[0] for l in lineas])

print("Contenido de dags/:", sorted(os.listdir(DAGS_DIR)))
print("Listo. Airflow lo toma en ~5 minutos.")


1) DAG OK: /home/jovyan/work/dags/pipeline_grupo07.py
2) .airflowignore escrito
3) dbt copiado a /home/jovyan/work/dags/dbt
4) .env escrito con: ['MINIO_BUCKET', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_ENDPOINT_URL', 'PGHOST', 'PGPORT', 'PGDATABASE', 'PGUSER', 'PGPASSWORD', 'URLHAUS_AUTH_KEY']
Contenido de dags/: ['.airflowignore', '.env', '__pycache__', 'dbt', 'pipeline_grupo07.py', 'scripts']
Listo. Airflow lo toma en ~5 minutos.


## 5. Verificar la capa gold en PostgreSQL

Corré esta celda después de que el DAG termine en verde: `publicar_gold` crea y carga `fct_amenazas_mensual` (por tipo de amenaza) y `fct_familias_mensual` (por familia de malware y tipo de archivo) en el warehouse, las que lee Metabase. Las dos tienen que sumar la misma cantidad de amenazas.

In [9]:
# Después de correr el DAG en Airflow: verificar que las dos tablas gold llegaron a PostgreSQL
from sqlalchemy import create_engine, text
from entorno import pg_url
with create_engine(pg_url()).connect() as c:
    print(c.execute(text("select table_schema, table_name from information_schema.tables "
                         "where table_name in ('fct_amenazas_mensual', 'fct_familias_mensual')")).fetchall())
    for tabla in ("fct_amenazas_mensual", "fct_familias_mensual"):
        print(tabla, c.execute(text(f"select count(*), sum(cantidad_amenazas) from {tabla}")).one())
    print("Top familias:")
    for fila in c.execute(text("select familia_malware, sum(cantidad_amenazas) as n from fct_familias_mensual "
                               "group by 1 order by n desc limit 10")):
        print("  ", fila)
    print("Tipos de archivo:")
    for fila in c.execute(text("select tipo_archivo, sum(cantidad_amenazas) as n from fct_familias_mensual "
                               "group by 1 order by n desc limit 10")):
        print("  ", fila)

[('grupo07', 'fct_amenazas_mensual'), ('grupo07', 'fct_familias_mensual')]
fct_amenazas_mensual (94, Decimal('189756'))
fct_familias_mensual (2209, Decimal('189756'))
Top familias:
   ('SmartLoader', Decimal('37928'))
   ('SalatStealer', Decimal('33587'))
   ('zapret', Decimal('24401'))
   ('sin_familia', Decimal('21493'))
   ('LuaJIT-loader', Decimal('18387'))
   ('mirai', Decimal('18120'))
   ('Mozi', Decimal('17501'))
   ('FakeGit', Decimal('2677'))
   ('CoinMiner', Decimal('1779'))
   ('CheatSheet', Decimal('1474'))
Tipos de archivo:
   ('desconocido', Decimal('84553'))
   ('zip', Decimal('68668'))
   ('elf', Decimal('23426'))
   ('exe', Decimal('7447'))
   ('sh', Decimal('1173'))
   ('pdf', Decimal('1101'))
   ('ascii', Decimal('1057'))
   ('js', Decimal('663'))
   ('rar', Decimal('584'))
   ('jar', Decimal('274'))
